# Исследование методов поиска кратчайшего пути в графах с распределением Кокса

## 1. Изучение распределения Кокса

### 1.1. Введение
**Распределение Кокса (Coxian distribution)** – это фазовая случайная величина, широко используемая в теории массового обслуживания и транспортном моделировании для описания времени обслуживания или интервалов между событиями. Оно представляет собой последовательность экспоненциально распределённых фаз, после каждой из которых с некоторой вероятностью процесс либо завершается, либо переходит к следующей фазе.

**Математическое описание:**
Распределение Кокса порядка $k$ задаётся:
- интенсивностями фаз $\mu_1, \mu_2, \dots, \mu_k$ ($\mu_i > 0$);
- вероятностями перехода $p_1, p_2, \dots, p_{k-1}$ ($0 \le p_i \le 1$), где $p_i$ – вероятность продолжения после $i$-й фазы, а $1-p_i$ – вероятность завершения на $i$-й фазе.
Фазы проходятся строго последовательно: начинаем с фазы 1, после её окончания с вероятностью $p_1$ переходим к фазе 2, иначе завершаемся. Если перешли к фазе 2, то после её окончания с вероятностью $p_2$ переходим к фазе 3 и т.д. После фазы $k$ процесс всегда завершается (т.е. $p_k = 0$).

Время реализации – сумма длительности пройденных фаз. Каждая фаза имеет экспоненциальное распределение со средним $1/\mu_i$.

**Свойства:**
- Гибкость: позволяет аппроксимировать различные распределения с коэффициентом вариации больше 1.
- Математическое ожидание:
  $
  E[T] = \sum_{i=1}^{k} \frac{1}{\mu_i} \prod_{j=1}^{i-1} p_j
  $
- Дисперсия вычисляется через вторые моменты экспоненциальных фаз и ковариации.

**Применение в транспортных задачах:**
Распределение Кокса используется для моделирования времени проезда по участку дороги с учётом светофоров, заторов, случайных событий. Фазы могут соответствовать состояниям транспортного потока (свободный, плотный, остановка). Изменение параметров во времени позволяет описывать динамику трафика.

## 2. Формирование математической модели


### 2.1. Модель графа
- Граф $G = (V, E)$ – ориентированный или неориентированный, заданного размера. Топология может быть фиксированной (например, сетка) или случайной (например, связный граф с заданным числом рёбер).
- Каждому ребру $e \in E$ сопоставлено распределение Кокса с параметрами $(\boldsymbol{\mu}, \boldsymbol{p})$, которые могут зависеть от времени $t$ (например, интенсивности фаз меняются по заданному закону). В простейшем случае параметры фиксированы.
- Вес ребра в момент времени $t$ – это случайная величина, генерируемая из соответствующего распределения. Динамика реализуется либо **независимым пересчётом** весов через фиксированные интервалы (дискретное время), либо **обновлением** веса при каждом обращении к ребру (как если бы мы измеряли его заново).



### 2.2. Генерация весов из распределения Кокса
Для моделирования одного значения:
1. Установить текущую фазу $i = 1$, суммарное время $T = 0$.
2. Пока $i \le k$:
   - Сгенерировать экспоненциальную величину $x \sim \text{Exp}(\mu_i)$.
   - $T \leftarrow T + x$.
   - Если $i = k$ или с вероятностью $1-p_i$ завершить, иначе $i \leftarrow i+1$.
3. Вернуть $T$.

Для повышения эффективности можно использовать прямое моделирование через матрицу переходов, но для учебных целей достаточно приведённого алгоритма.

#### 2.3. Влияние параметров распределения на топологию графа
- При фиксированной топологии параметры определяют распределение длин путей. Изменяя параметры (например, увеличивая число фаз или уменьшая интенсивности), можно моделировать различные условия – от почти детерминированных до высоковариабельных задержек.
- Влияние на топологию: если на некоторых рёбрах дисперсия велика, то пути, включающие такие рёбра, становятся менее предсказуемыми. Это может привести к тому, что кратчайший путь в среднем может не совпадать с путём, минимизирующим квантиль или максиминный критерий.

## 3. Реализация алгоритмов поиска пути

### 3.1. Классические алгоритмы
Для графов с неотрицательными весами (веса из распределения Кокса всегда положительны) подходят:
- **Алгоритм Дейкстры** – находит кратчайший путь от источника ко всем вершинам за $O(|E| + |V|\log|V|)$ при использовании приоритетной очереди.
- **Алгоритм Беллмана–Форда** – работает и с отрицательными весами, но здесь избыточен; его можно использовать для проверки корректности.
- **A*** – с эвристикой (например, евклидово расстояние в координатной сетке) ускоряет поиск.

В условиях динамически меняющихся весов классические алгоритмы применяются к текущему состоянию графа. Для каждого момента времени $t$ можно заново запустить Дейкстру и получить путь, оптимальный для данного набора весов.

### 3.2. Учёт динамики весов
Если веса меняются во времени, оптимальный путь, найденный в момент старта, может перестать быть таковым в процессе движения. Возможные подходы:
- **Статический подход** – используем веса, актуальные на момент начала движения. Просто, но неадаптивно.
- **Адаптивный подход** – после каждого шага (или периодически) пересчитываем путь с учётом обновлённых весов. По сути, это многократный запуск Дейкстры.

## 4. Разработка модификации алгоритма

### 4.1. Предлагаемая модификация: Адаптивный вероятностный Дейкстра
Классический Дейкстра оперирует детерминированными весами. В условиях, когда веса – случайные величины, изменяющиеся во времени, целесообразно учитывать их распределение. Предлагается модификация, которая на каждом шаге использует не только текущие значения, но и **прогноз** ожидаемых весов на оставшихся рёбрах, что позволяет выбирать путь, минимизирующий ожидаемое полное время до цели с учётом возможных будущих изменений.

**Идея:**
- В каждый момент времени $t$ мы знаем текущие веса рёбер (измеренные или предсказанные).
- Для каждого ребра доступны параметры распределения Кокса, позволяющие вычислить математическое ожидание $E[w]$.
- Модифицируем релаксацию в алгоритме Дейкстры: вместо сравнения с точным весом используем **ожидаемое время прибытия** в вершину, которое складывается из уже накопленного детерминированного времени (по пройденным рёбрам) и ожидаемого времени на оставшемся пути. Но поскольку оставшийся путь тоже содержит неопределённость, можно применить принцип оптимальности Беллмана для ожидаемого значения.

**Реализация:**
1. Для каждой вершины храним оценку $dist[v]$ – ожидаемое минимальное время от источника до $v$ с учётом всех неопределённостей.
2. Начальное значение $dist[source] = 0$, для остальных $+\infty$.
3. Используем приоритетную очередь по $dist$.
4. При обработке вершины $u$ рассматриваем все исходящие рёбра $(u,v)$. Для каждого ребра известно его распределение с параметрами, но фактическое значение в момент прохода неизвестно. Однако мы можем оценить вклад ребра как его математическое ожидание $E[w_{uv}]$. Тогда новая оценка для $v$: $dist[u] + E[w_{uv}]$. Если она меньше текущей $dist[v]$, обновляем.
5. После нахождения пути от источника до цели мы получаем путь, минимизирующий ожидаемое полное время (в предположении, что веса независимы и не меняются во времени, кроме как через пересчёт).

**Учёт динамики:**
После того как мы реально проходим по ребру, его вес становится известным (фактическое время). Накопленное время становится детерминированным. При достижении очередной вершины мы можем перезапустить алгоритм с этой вершины как с нового источника, используя текущие веса остальных рёбер (возможно, обновлённые). Это сочетает адаптивность и учёт распределений.

### 4.2. Альтернативная модификация: Робастный путь
Минимизация не только среднего, но и некоторого квантиля (например, 95-го процентиля) распределения полного времени пути. Для этого нужно уметь вычислять распределение суммы случайных величин (свёртку фазовых распределений), что сложно, но можно аппроксимировать методом Монте-Карло. Однако для учебной работы достаточно варианта с ожиданием.

## 5. Тестирование и анализ результатов

### 5.1. План экспериментов
- **Генерация графов:**
  - Размер: от 10 до 100 вершин.
  - Топология: полносвязный, решётка, случайный связный граф.
  - Параметры распределения Кокса: варьируем число фаз (1–5), интенсивности (от 0.1 до 10), вероятности продолжения (0.2–0.9).
- **Сценарии динамики:**
  - Статические веса (одна реализация).
  - Веса меняются каждую секунду (дискретное время) независимо.
  - Веса меняются по тренду (например, интенсивность фазы линейно растёт со временем).
- **Сравниваемые алгоритмы:**
  - Классический Дейкстра (запускается однократно в начальный момент).
  - Адаптивный Дейкстра (перезапуск в каждой вершине с текущими весами).
  - Модифицированный вероятностный Дейкстра (с использованием мат. ожиданий).
- **Метрики:**
  - Фактическое время прохождения пути (сумма реализованных весов пройденных рёбер).
  - Отклонение от оптимального (если бы все будущие веса были известны).
  - Время работы алгоритма.
  - Устойчивость найденного пути (частота, с которой путь изменяется при пересчёте).

### 5.2. Анализ
- Ожидается, что адаптивный перезапуск Дейкстры даёт лучшие результаты по фактическому времени, чем однократный запуск, особенно при высокой вариабельности весов.
- Вероятностный Дейкстра (с мат. ожиданием) может давать компромисс между вычислительной сложностью и качеством пути, особенно если распределения имеют малую дисперсию.
- При увеличении числа фаз и дисперсии весов преимущество адаптивных методов должно расти.